In [13]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pathlib import Path
import json
import os
import re

# ============================================================
# PATHS
# ============================================================

# INPUT_FILE = Path(
#     r"/Users/ferrisramadan/Desktop/ai_analysis/related_occs/High_VeryHigh_AND_Medium_Related_Occs_Final.xlsx"
# )
INPUT_FILE = Path(
    r"G:\.shortcut-targets-by-id\1KKrBsNyhwzbYus9ExtUkUd_3jE4VZ4W6\AI_Analysis\Arielle's Stuff\High_VeryHigh_AND_Medium_Related_Occs_Final.xlsx"
)

# OEWS_FILE = Path(
#     r"/Users/ferrisramadan/Desktop/ai_analysis/ai_impact_analysis/ai_skills/OEWS/Arizona OEWS 2024.xlsx"
# )
OEWS_FILE = Path(
    r"C:\Users\301533\Downloads\Arizona 2024 OEWS.xlsx"
)

OUT_HTML = Path(
    r"C:\Users\301533\Documents\GitHub\AI\AI Website\aipathways.github.io\assets\sankey.html"
)

# ============================================================
# SETTINGS
# ============================================================

SOURCE_EXPOSURE_GROUPS = ["High", "Very High"]
LOWER_EXPOSURE_GROUPS = ["Medium", "Low", "Very Low"]

TOP_DESTINATIONS_PER_VIEW = 10

# Build a bigger pool first so direct/intermediary views still have enough options.
MAX_DIRECT_OCCS = 20
MAX_BRIDGE_OCCS = 8
MAX_BRIDGE_OCCS_IF_FEW_DIRECT = 10
DIRECT_COUNT_THRESHOLD = 8
MAX_DEST_PER_BRIDGE = 7

LOW_WAGE_CUTOFF = 50000
MID_WAGE_CUTOFF = 100000

# ============================================================
# CLEANER COLOR SYSTEM
# ============================================================

DARK = "#333333"

# Main structure colors
SOURCE_COLOR = "#075877"          # dark turquoise
DIRECT_COLOR = "#F06E6C"          # coral direct pathway, intentionally not same as exposure
INTERMEDIARY_COLOR = "#F79732"    # carrot intermediary pathway
WAGE_COLOR = "#A39B98"            # storm gray wage node

# Destination exposure colors. These are used for the final pathway segment
# and the destination nodes.
EXPOSURE_COLORS = {
    "Very Low": "#17C3B2",   # patina green
    "Low": "#8ED3F5",        # sky blue
    "Medium": "#75618C",     # violet
    "High": "#BB638F",       # magenta
    "Very High": "#495568",  # dark gray
}

EDUCATION_RANK = {
    "no formal education": 1,
    "no formal educational credential": 1,
    "high school diploma": 2,
    "high school diploma or equivalent": 2,
    "some college, no degree": 3,
    "postsecondary non-degree": 4,
    "postsecondary nondegree": 4,
    "postsecondary nondegree award": 4,
    "postsecondary non-degree award": 4,
    "associate's degree": 5,
    "associates degree": 5,
    "bachelor's degree": 6,
    "bachelors degree": 6,
    "master's degree": 7,
    "masters degree": 7,
    "doctoral or professional degree": 8,
    "doctorate or professional degree": 8,
}

# ============================================================
# HELPERS
# ============================================================

def clean_soc(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip().replace(".0", "")

    # Fix Excel date corruption for 11-xxxx occupations.
    if s.startswith("Nov-"):
        nov_map = {
            "11": "11-2011", "21": "11-2021", "22": "11-2022",
            "32": "11-2032", "33": "11-2033", "12": "11-3012",
            "13": "11-3013", "31": "11-3031", "51": "11-3051",
            "61": "11-3061", "71": "11-3071", "41": "11-9041",
            "72": "11-9072", "81": "11-9081", "79": "11-9179",
            "99": "11-9199",
        }
        return nov_map.get(s.replace("Nov-", "").strip(), np.nan)

    return s


def clean_money(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    if s in ["", "-", "NA", "N/A", "nan", "None", "*", "**"]:
        return np.nan

    return pd.to_numeric(
        s.replace("$", "").replace(",", "").replace("−", "-"),
        errors="coerce"
    )


def annualize_hourly_wage(x):
    hourly = clean_money(x)

    if pd.isna(hourly):
        return np.nan

    return hourly * 2080


def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def clean_num(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    if s in ["", "-", "NA", "N/A", "nan", "None", "*", "**"]:
        return np.nan

    return pd.to_numeric(
        s.replace(",", "").replace("%", "").replace("−", "-"),
        errors="coerce"
    )


def clean_percent_to_decimal(x):
    if pd.isna(x):
        return np.nan

    raw = str(x).strip()
    val = clean_num(x)

    if pd.isna(val):
        return np.nan

    if "%" in raw:
        return val / 100

    if abs(val) <= 1:
        return val

    return val / 100


def clean_str(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def fmt_currency(x):
    if pd.isna(x):
        return "NA"
    return f"${x:,.0f}"


def fmt_num(x):
    if pd.isna(x):
        return "NA"
    return f"{x:,.0f}"


def fmt_percent(x):
    if pd.isna(x):
        return "NA"
    return f"{x:.1%}"


def fmt_score(x):
    if pd.isna(x):
        return "NA"
    return f"{x:.1f}"


def fmt_diff(x):
    if pd.isna(x):
        return "NA"

    sign = "+" if x >= 0 else "-"
    return f"{sign}${abs(x):,.0f}"


def wrap_label(label, width=30, max_lines=3):
    label = str(label)

    words = label.split()
    lines = []
    current = ""

    for word in words:
        test = (current + " " + word).strip()

        if len(test) <= width:
            current = test
        else:
            if current:
                lines.append(current)
            current = word

    if current:
        lines.append(current)

    if len(lines) > max_lines:
        lines = lines[:max_lines]
        lines[-1] = lines[-1][: max(0, width - 3)] + "..."

    return "<br>".join(lines)


def short(label, n=34):
    return wrap_label(label, width=n, max_lines=3)


def hex_to_rgba(hex_color, alpha):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i:i + 2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"


def make_positions(n, top=0.15, bottom=0.88):
    if n <= 1:
        return [0.50]
    return list(np.linspace(top, bottom, n))


def normalize_education(value):
    if pd.isna(value):
        return ""

    s = str(value).strip().lower()
    s = s.replace("’", "'").replace("–", "-").replace("—", "-")
    s = s.replace("nondegree", "non-degree")
    return " ".join(s.split())


def clean_education_label(value):
    if pd.isna(value) or str(value).strip() in ["", "nan", "None"]:
        return "Unknown"

    s = str(value).strip()

    replacements = {
        "no formal education": "No formal education",
        "no formal educational credential": "No formal credential",
        "high school diploma or equivalent": "High school diploma",
        "some college, no degree": "Some college, no degree",
        "postsecondary non-degree award": "Postsecondary nondegree",
        "postsecondary nondegree award": "Postsecondary nondegree",
        "associate's degree": "Associate's degree",
        "associates degree": "Associate's degree",
        "bachelor's degree": "Bachelor's degree",
        "bachelors degree": "Bachelor's degree",
        "master's degree": "Master's degree",
        "masters degree": "Master's degree",
        "doctoral or professional degree": "Doctoral or professional degree",
        "doctorate or professional degree": "Doctoral or professional degree",
    }

    return replacements.get(normalize_education(s), s)


def education_rank(value):
    return EDUCATION_RANK.get(normalize_education(value), np.nan)


def allowed_target_groups(source_exposure):
    if clean_str(source_exposure) in ["Very High", "High"]:
        return ["Medium", "Low", "Very Low"]
    return []


def dynamic_wage_thresholds(source_wage):
    """
    Similar wage rule:
    < $50k source wage: +/- 5%
    $50k to $100k source wage: +/- 10%
    > $100k source wage: +/- 15%
    """
    if pd.isna(source_wage):
        return np.nan, np.nan, "Unknown wage source"

    if source_wage < LOW_WAGE_CUTOFF:
        pct = 0.05
        rule_name = "under $50k"
    elif source_wage <= MID_WAGE_CUTOFF:
        pct = 0.10
        rule_name = "$50k to $100k"
    else:
        pct = 0.15
        rule_name = "over $100k"

    threshold = source_wage * pct
    return threshold, threshold, rule_name


def wage_bucket(target_wage, source_wage):
    if pd.isna(target_wage) or pd.isna(source_wage):
        return "Wage unknown"

    diff = target_wage - source_wage
    loss_threshold, gain_threshold, _ = dynamic_wage_thresholds(source_wage)

    if pd.isna(loss_threshold) or pd.isna(gain_threshold):
        return "Wage unknown"

    if diff > gain_threshold:
        return "Higher wage"

    if diff < -loss_threshold:
        return "Lower wage"

    return "Similar wage"


def wage_sort_rank(wage_group):
    return {
        "Higher wage": 1,
        "Similar wage": 2,
        "Lower wage": 3,
        "Wage unknown": 4,
    }.get(wage_group, 5)


def style_label(label, label_type):
    if label_type == "source":
        return f"<b><span style='font-size:13px; color:#111111'>{label}</span></b>"

    if label_type == "wage":
        return f"<b><span style='font-size:14px; color:#111111'>{label}</span></b>"

    if label_type == "destination":
        label = label.replace(
            "●",
            "<span style='color:#BB638F; font-size:15px'>●</span>"
        )
        return f"<b><span style='font-size:11.5px; color:#111111'>{label}</span></b>"

    return f"<span style='font-size:11px; color:#111111'>{label}</span>"


# ============================================================
# RELATIONSHIP HELPERS
# ============================================================

def get_best_compat(row):
    for col in [
        "Combined_Score_0_100",
        "Lightcast_Compatibility_Index",
        "Combined_Compat_Index",
        "Final_Compat_Index",
        "Weighted_Compat_Index",
        "Avg_Compat_Index",
        "Median_Compat_Index",
        "Max_Compat_Index",
        "Min_Compat_Index",
        "Gateway_Compat_Index",
        "Destination_Compat_Index",
        "ONET_Index",
    ]:
        if col in row.index and pd.notna(row[col]):
            return pd.to_numeric(row[col], errors="coerce")

    return np.nan


def relationship_strength(source_type, onet_tier, score):
    source_type = clean_str(source_type)
    onet_tier = clean_str(onet_tier)

    if source_type == "Both":
        return "Strongest match"

    if onet_tier == "Primary-Short":
        return "Strong match"

    if onet_tier == "Primary-Long":
        return "Moderate match"

    if onet_tier == "Supplemental":
        return "Broader related match"

    if pd.notna(score):
        if score >= 95:
            return "Strong match"
        if score >= 90:
            return "Moderate match"
        return "Broader related match"

    return "Related match"


def relationship_weight(strength):
    return {
        "Strongest match": 5,
        "Strong match": 4,
        "Moderate match": 2.5,
        "Broader related match": 1.5,
        "Related match": 1,
    }.get(strength, 1)


def relationship_rank(strength):
    return {
        "Strongest match": 1,
        "Strong match": 2,
        "Moderate match": 3,
        "Broader related match": 4,
        "Related match": 5,
    }.get(strength, 6)


def onet_rank(tier):
    return {
        "Primary-Short": 1,
        "Primary-Long": 2,
        "Supplemental": 3,
    }.get(clean_str(tier), 9)


# ============================================================
# LOAD DATA
# ============================================================

xl = pd.ExcelFile(INPUT_FILE)

print("Workbook sheets:", xl.sheet_names)

pathway_sheet = "Related_Occs_Final"

final = pd.read_excel(INPUT_FILE, sheet_name=pathway_sheet)
final.columns = final.columns.astype(str).str.strip()

ai_econ = pd.read_excel(INPUT_FILE, sheet_name="AI_Economic")
ai_econ.columns = ai_econ.columns.astype(str).str.strip()

print("Pathway sheet used:", pathway_sheet)
print("Rows in pathway sheet:", len(final))
print("Rows in AI_Economic:", len(ai_econ))

final = final.rename(columns={
    "Pathway Type": "Original_Pathway_Type",
    "Source SOC": "Source_SOC",
    "Source Occupation": "Source_Occupation",
    "Source AI Exposure": "Source_AI_Exposure_Group",
    "Gateway Rank": "Gateway_Rank",
    "Gateway SOC": "Gateway_SOC",
    "Gateway Occupation": "Gateway_Occupation",
    "Gateway AI Exposure": "Gateway_AI_Exposure_Group",
    "Related Occupation SOC": "Target_SOC",
    "Related Occupation": "Target_Occupation",
    "Related Occupation AI Exposure": "Target_AI_Exposure_Group",
    "Related Occupation O*NET Tier": "ONET_Tier",
    "Related Occupation O*NET Index": "ONET_Index",
    "Lightcast Compatibility Index": "Lightcast_Compatibility_Index",
    "Lightcast Score": "Lightcast_Score",
    "Related Occupation O*NET Score": "ONET_Score",
    "Combined Lightcast + O*NET Score": "Combined_Compat_Index",
    "Combined Score 0-100": "Combined_Score_0_100",
})

required_final_cols = [
    "Source_SOC",
    "Source_Occupation",
    "Source_AI_Exposure_Group",
    "Target_SOC",
    "Target_Occupation",
    "Target_AI_Exposure_Group",
]

missing_final_cols = [c for c in required_final_cols if c not in final.columns]

if missing_final_cols:
    print("Available final columns:")
    print(list(final.columns))
    raise ValueError(f"Missing required columns in pathway sheet: {missing_final_cols}")

for col in ["Source_SOC", "Target_SOC", "Gateway_SOC"]:
    if col in final.columns:
        final[col] = final[col].apply(clean_soc)

ai_econ["SOC"] = ai_econ["SOC"].apply(clean_soc)

if "Original_Pathway_Type" in final.columns:
    final["Original_Pathway_Type"] = final["Original_Pathway_Type"].apply(clean_str)
else:
    final["Original_Pathway_Type"] = ""

for col in [
    "Combined_Score_0_100",
    "Combined_Compat_Index",
    "Lightcast_Compatibility_Index",
    "Lightcast_Score",
    "ONET_Score",
    "ONET_Index",
    "Gateway_Rank",
]:
    if col in final.columns:
        final[col] = pd.to_numeric(final[col], errors="coerce")

if "Source_Type" not in final.columns:
    final["Source_Type"] = np.where(
        final["Lightcast_Compatibility_Index"].notna() & final["ONET_Index"].notna(),
        "Both",
        np.where(final["Lightcast_Compatibility_Index"].notna(), "Lightcast", "O*NET")
    )

final["Source_Type"] = final["Source_Type"].apply(clean_str)
final["ONET_Tier"] = final["ONET_Tier"].apply(clean_str)
final["Score"] = final.apply(get_best_compat, axis=1)

final["Relationship_Strength"] = final.apply(
    lambda r: relationship_strength(r.get("Source_Type"), r.get("ONET_Tier"), r.get("Score")),
    axis=1
)

final["Relationship_Weight"] = final["Relationship_Strength"].apply(relationship_weight)
final["Relationship_Rank"] = final["Relationship_Strength"].apply(relationship_rank)
final["ONET_Tier_Rank"] = final["ONET_Tier"].apply(onet_rank)

econ = (
    ai_econ[[
        "SOC",
        "AI_Occupation_Title",
        "AI_Exposure_Group",
        "AI_Annual_Total_Openings",
        "AI_Annual_Percent_Change",
        "AI_Education_Value",
        "AI_Work_Experience_Value",
        "AI_50th_Percentile_Wage",
    ]]
    .rename(columns={
        "AI_Occupation_Title": "Occupation Title",
        "AI_Exposure_Group": "Exposure",
        "AI_Annual_Total_Openings": "Openings",
        "AI_Annual_Percent_Change": "Growth",
        "AI_Education_Value": "Education_Value",
        "AI_Work_Experience_Value": "Work_Experience_Value",
        "AI_50th_Percentile_Wage": "Median_Wage",
    })
    .dropna(subset=["SOC"])
    .drop_duplicates(subset=["SOC"])
    .copy()
)

econ["Median_Wage"] = econ["Median_Wage"].apply(clean_money)
econ["Openings"] = econ["Openings"].apply(clean_num)
econ["Growth"] = econ["Growth"].apply(clean_percent_to_decimal)
econ["Employment"] = np.nan

# ============================================================
# ADD ARIZONA OEWS 2024 MEDIAN WAGE FALLBACK
# ============================================================

if OEWS_FILE.exists():
    oews = pd.read_excel(OEWS_FILE, skiprows=1, sheet_name="Annual")
    oews.columns = oews.columns.astype(str).str.strip()

    oews_soc_col = first_existing_col(oews, [
        "SOC Code1",
        "SOC Code",
        "SOC",
        "OCC_CODE",
        "occ_code",
        "Occupation Code",
    ])

    oews_median_col = first_existing_col(oews, [
        "50th Percentile Wage",
        "Median Wage",
        "Median",
        "A_MEDIAN",
        "H_MEDIAN",
    ])

    if oews_soc_col is None or oews_median_col is None:
        print("OEWS file found, but required SOC or median wage column was not found.")
        print("OEWS columns:", list(oews.columns))
    else:
        oews_wage = (
            oews[[oews_soc_col, oews_median_col]]
            .rename(columns={
                oews_soc_col: "SOC",
                oews_median_col: "OEWS_50th_Percentile_Hourly_Wage",
            })
            .copy()
        )

        oews_wage["SOC"] = oews_wage["SOC"].apply(clean_soc)
        oews_wage["OEWS_Annual_Median_Wage"] = (
            oews_wage["OEWS_50th_Percentile_Hourly_Wage"].apply(annualize_hourly_wage)
        )

        oews_wage = (
            oews_wage
            .dropna(subset=["SOC"])
            .drop_duplicates(subset=["SOC"])
            [["SOC", "OEWS_Annual_Median_Wage"]]
        )

        econ = econ.merge(oews_wage, on="SOC", how="left")

        before_missing = int(econ["Median_Wage"].isna().sum())
        econ["Median_Wage"] = econ["Median_Wage"].fillna(econ["OEWS_Annual_Median_Wage"])
        after_missing = int(econ["Median_Wage"].isna().sum())

        print("OEWS wage fallback applied.")
        print("Missing wages before OEWS fallback:", before_missing)
        print("Missing wages after OEWS fallback:", after_missing)
else:
    print("OEWS wage file not found at:", OEWS_FILE)

econ_lookup = econ.set_index("SOC").to_dict("index")


# ============================================================
# BUILD WEBSITE OCCUPATION ID LOOKUP FOR DUPLICATE SOCS
# Uses website data.json so titles/ids match the site exactly
# ============================================================

DATA_JSON = Path(
    r"C:\Users\301533\Documents\GitHub\AI\AI Website\aipathways.github.io\data.json"
)

with open(DATA_JSON, "r", encoding="utf-8") as f:
    website_occ = json.load(f)

ferris_occ = pd.DataFrame([
    {
        "id": occ.get("id"),
        "title": occ.get("title"),
        "soc": clean_soc(occ.get("soc")),
        "exposure": occ.get("exposure")
    }
    for occ in website_occ
])

ferris_occ = ferris_occ.dropna(subset=["id", "title", "soc"]).drop_duplicates()

ferris_occ.head()

def econ_value(soc, field):
    return econ_lookup.get(str(soc), {}).get(field, np.nan)


def requires_more_education(source_soc, target_soc):
    source_edu = econ_value(source_soc, "Education_Value")
    target_edu = econ_value(target_soc, "Education_Value")

    source_rank = education_rank(source_edu)
    target_rank = education_rank(target_edu)

    if pd.isna(source_rank) or pd.isna(target_rank):
        return False

    return target_rank > source_rank


def education_change_label(source_soc, target_soc):
    source_edu = econ_value(source_soc, "Education_Value")
    target_edu = econ_value(target_soc, "Education_Value")

    source_rank = education_rank(source_edu)
    target_rank = education_rank(target_edu)

    if pd.isna(source_rank) or pd.isna(target_rank):
        return "Unknown"

    diff = int(target_rank - source_rank)

    if diff > 0:
        return f"+{diff} level"

    if diff < 0:
        return f"{abs(diff)} level lower"

    return "Same level"


def build_hover_text(
    title,
    soc,
    source_soc=None,
    exposure=None,
    score=None,
    wage_group=None,
    pathway=None,
    bridge=None,
    source_type=None,
    onet_tier=None,
    relationship=None,
    role_label="Occupation"
):
    """
    Short hover text. Full details are in the ranked table.
    """
    e = econ_lookup.get(str(soc), {})
    source_e = econ_lookup.get(str(source_soc), {}) if source_soc else {}

    dest_wage = e.get("Median_Wage")
    source_wage = source_e.get("Median_Wage")
    wage_diff = dest_wage - source_wage if pd.notna(dest_wage) and pd.notna(source_wage) else np.nan

    compat_parts = []
    if relationship:
        compat_parts.append(str(relationship))
    if onet_tier:
        compat_parts.append(str(onet_tier))
    if score is not None and pd.notna(score):
        compat_parts.append(f"score {fmt_score(score)}")

    compat_text = ", ".join(compat_parts) if compat_parts else "NA"

    edu_text = clean_education_label(e.get("Education_Value"))

    if source_soc:
        edu_change = education_change_label(source_soc, soc)
    else:
        edu_change = "NA"

    pathway_text = ""
    if bridge:
        pathway_text = f"<br>Pathway: via {bridge}"
    elif pathway:
        pathway_text = f"<br>Pathway: {pathway}"

    return (
        f"<b>{title}</b><br>"
        f"SOC: {soc} | Exposure: {exposure or 'NA'}<br>"
        f"Compatibility: {compat_text}<br>"
        f"Wage: {fmt_currency(dest_wage)}"
        f"{f' ({fmt_diff(wage_diff)} vs source)' if source_soc else ''} | {wage_group or 'NA'}<br>"
        f"Education: {edu_text} | Change: {edu_change}"
        f"{pathway_text}"
    )


# ============================================================
# BUILD SOURCE OPTIONS
# ============================================================

source_options = (
    final[
        (final["Source_AI_Exposure_Group"].isin(SOURCE_EXPOSURE_GROUPS)) &
        (final["Source_SOC"].notna())
    ][["Source_SOC", "Source_Occupation", "Source_AI_Exposure_Group"]]
    .drop_duplicates()
    .sort_values(["Source_AI_Exposure_Group", "Source_Occupation"])
    .copy()
)

print("High and Very High source occupations included:", len(source_options))


# ============================================================
# BUILD EDGE DATA
# ============================================================

def build_edges_for_source(source_soc):
    source_rows = final[final["Source_SOC"].astype(str) == str(source_soc)].copy()

    if source_rows.empty:
        return pd.DataFrame()

    source_title = source_rows["Source_Occupation"].dropna().iloc[0]
    source_exposure = source_rows["Source_AI_Exposure_Group"].dropna().iloc[0]
    source_wage = econ_value(source_soc, "Median_Wage")
    target_groups = allowed_target_groups(source_exposure)

    if not target_groups:
        return pd.DataFrame()

    all_rows = []

    # Direct pathways
    direct = final[
        (final["Source_SOC"].astype(str) == str(source_soc)) &
        (final["Source_AI_Exposure_Group"] == source_exposure) &
        (final["Target_AI_Exposure_Group"].isin(target_groups))
    ].copy()

    if not direct.empty:
        direct["Wage"] = direct["Target_SOC"].map(lambda s: econ_value(s, "Median_Wage"))
        direct["Growth"] = direct["Target_SOC"].map(lambda s: econ_value(s, "Growth"))
        direct["Wage_Group"] = direct["Target_SOC"].map(
            lambda s: wage_bucket(econ_value(s, "Median_Wage"), source_wage)
        )
        direct["Wage_Rank"] = direct["Wage_Group"].apply(wage_sort_rank)

        direct = (
            direct
            .drop_duplicates(subset=["Target_SOC"])
            .sort_values(
                ["ONET_Tier_Rank", "Relationship_Rank", "Score", "Wage_Rank", "Wage", "Growth"],
                ascending=[True, True, False, True, False, False]
            )
            .head(MAX_DIRECT_OCCS)
        )

        for _, row in direct.iterrows():
            dest_soc = str(row["Target_SOC"])
            dest_wage = econ_value(dest_soc, "Median_Wage")

            all_rows.append({
                "Source_SOC": source_soc,
                "Source_Occupation": source_title,
                "Source_AI_Exposure_Group": source_exposure,
                "Pathway_View": "direct",
                "Pathway_Label": "Direct",
                "Bridge_SOC": "",
                "Bridge_Occupation": "",
                "Destination_SOC": dest_soc,
                "Destination_Occupation": row["Target_Occupation"],
                "Destination_AI_Exposure_Group": row["Target_AI_Exposure_Group"],
                "Score": row["Score"],
                "Source_Type": row.get("Source_Type", ""),
                "ONET_Tier": row.get("ONET_Tier", ""),
                "Relationship_Strength": row.get("Relationship_Strength", "Related match"),
                "Relationship_Weight": row.get("Relationship_Weight", 1),
                "Wage_Group": wage_bucket(dest_wage, source_wage),
                "Requires_More_Education": requires_more_education(source_soc, dest_soc),
            })

    direct_count = len(direct) if not direct.empty else 0
    bridge_limit = MAX_BRIDGE_OCCS_IF_FEW_DIRECT if direct_count < DIRECT_COUNT_THRESHOLD else MAX_BRIDGE_OCCS

    # Candidate intermediary roles are lower-exposure occupations directly related to the source.
    possible_bridges = final[
        (final["Source_SOC"].astype(str) == str(source_soc)) &
        (final["Source_AI_Exposure_Group"] == source_exposure) &
        (final["Target_AI_Exposure_Group"].isin(target_groups))
    ].copy()

    if not possible_bridges.empty:
        possible_bridges["Gateway_Wage"] = possible_bridges["Target_SOC"].map(
            lambda s: econ_value(s, "Median_Wage")
        )
        possible_bridges["Gateway_Growth"] = possible_bridges["Target_SOC"].map(
            lambda s: econ_value(s, "Growth")
        )
        possible_bridges["Gateway_Wage_Group"] = possible_bridges["Target_SOC"].map(
            lambda s: wage_bucket(econ_value(s, "Median_Wage"), source_wage)
        )
        possible_bridges["Gateway_Wage_Rank"] = possible_bridges["Gateway_Wage_Group"].apply(wage_sort_rank)

        bridges = (
            possible_bridges[[
                "Target_SOC",
                "Target_Occupation",
                "Target_AI_Exposure_Group",
                "Score",
                "Source_Type",
                "ONET_Tier",
                "Relationship_Strength",
                "Relationship_Weight",
                "Relationship_Rank",
                "ONET_Tier_Rank",
                "Gateway_Wage",
                "Gateway_Growth",
                "Gateway_Wage_Rank",
            ]]
            .rename(columns={
                "Target_SOC": "Gateway_SOC",
                "Target_Occupation": "Gateway_Occupation",
                "Target_AI_Exposure_Group": "Gateway_AI_Exposure_Group",
                "Score": "Gateway_Compat_Index",
                "Source_Type": "Gateway_Source_Type",
                "ONET_Tier": "Gateway_ONET_Tier",
                "Relationship_Strength": "Gateway_Relationship_Strength",
                "Relationship_Weight": "Gateway_Relationship_Weight",
                "Relationship_Rank": "Gateway_Relationship_Rank",
                "ONET_Tier_Rank": "Gateway_ONET_Tier_Rank",
            })
            .drop_duplicates(subset=["Gateway_SOC"])
            .sort_values(
                [
                    "Gateway_ONET_Tier_Rank",
                    "Gateway_Relationship_Rank",
                    "Gateway_Compat_Index",
                    "Gateway_Wage_Rank",
                    "Gateway_Wage",
                    "Gateway_Growth",
                ],
                ascending=[True, True, False, True, False, False]
            )
            .head(bridge_limit)
        )

        for _, bridge in bridges.iterrows():
            bridge_soc = str(bridge["Gateway_SOC"])

            dests = final[
                (final["Source_SOC"].astype(str) == bridge_soc) &
                (final["Target_AI_Exposure_Group"].isin(target_groups))
            ].copy()

            if dests.empty:
                continue

            dests = dests[dests["Target_SOC"].astype(str) != str(source_soc)].copy()
            dests = dests[dests["Target_SOC"].astype(str) != bridge_soc].copy()

            if dests.empty:
                continue

            dests["Destination_Wage"] = dests["Target_SOC"].map(
                lambda s: econ_value(s, "Median_Wage")
            )
            dests["Destination_Growth"] = dests["Target_SOC"].map(
                lambda s: econ_value(s, "Growth")
            )
            dests["Wage_Group"] = dests["Target_SOC"].map(
                lambda s: wage_bucket(econ_value(s, "Median_Wage"), source_wage)
            )
            dests["Wage_Rank"] = dests["Wage_Group"].apply(wage_sort_rank)

            dests = (
                dests
                .drop_duplicates(subset=["Target_SOC"])
                .sort_values(
                    [
                        "ONET_Tier_Rank",
                        "Relationship_Rank",
                        "Score",
                        "Wage_Rank",
                        "Destination_Wage",
                        "Destination_Growth",
                    ],
                    ascending=[True, True, False, True, False, False]
                )
                .head(MAX_DEST_PER_BRIDGE)
            )

            for _, dest in dests.iterrows():
                dest_soc = str(dest["Target_SOC"])
                dest_wage = econ_value(dest_soc, "Median_Wage")

                all_rows.append({
                    "Source_SOC": source_soc,
                    "Source_Occupation": source_title,
                    "Source_AI_Exposure_Group": source_exposure,
                    "Pathway_View": "bridge",
                    "Pathway_Label": "Intermediary",
                    "Bridge_SOC": bridge_soc,
                    "Bridge_Occupation": bridge["Gateway_Occupation"],
                    "Destination_SOC": dest_soc,
                    "Destination_Occupation": dest["Target_Occupation"],
                    "Destination_AI_Exposure_Group": dest["Target_AI_Exposure_Group"],
                    "Score": dest["Score"],
                    "Source_Type": dest.get("Source_Type", ""),
                    "ONET_Tier": dest.get("ONET_Tier", ""),
                    "Relationship_Strength": dest.get("Relationship_Strength", "Related match"),
                    "Relationship_Weight": dest.get("Relationship_Weight", 1),
                    "Wage_Group": wage_bucket(dest_wage, source_wage),
                    "Requires_More_Education": requires_more_education(source_soc, dest_soc),
                })

    edges = pd.DataFrame(all_rows)

    if not edges.empty:
        edges["Destination_Wage"] = edges["Destination_SOC"].map(
            lambda s: econ_value(s, "Median_Wage")
        )
        edges["Destination_Growth"] = edges["Destination_SOC"].map(
            lambda s: econ_value(s, "Growth")
        )
        edges["Destination_Openings"] = edges["Destination_SOC"].map(
            lambda s: econ_value(s, "Openings")
        )
        edges["Relationship_Rank"] = edges["Relationship_Strength"].apply(relationship_rank)
        edges["ONET_Tier_Rank"] = edges["ONET_Tier"].apply(onet_rank)
        edges["Wage_Rank"] = edges["Wage_Group"].apply(wage_sort_rank)

        edges = (
            edges
            .sort_values(
                [
                    "ONET_Tier_Rank",
                    "Relationship_Rank",
                    "Score",
                    "Wage_Rank",
                    "Destination_Wage",
                    "Destination_Growth",
                    "Destination_Openings",
                ],
                ascending=[True, True, False, True, False, False, False]
            )
            .copy()
        )

    return edges


# ============================================================
# RANKING AND TABLE HELPERS
# ============================================================

def select_top_destinations(edges, n=10):
    if edges.empty:
        return edges

    ranked = edges.copy()

    ranked["Destination_Wage"] = ranked["Destination_SOC"].map(
        lambda s: econ_value(s, "Median_Wage")
    )
    ranked["Destination_Growth"] = ranked["Destination_SOC"].map(
        lambda s: econ_value(s, "Growth")
    )
    ranked["Destination_Openings"] = ranked["Destination_SOC"].map(
        lambda s: econ_value(s, "Openings")
    )
    ranked["Relationship_Rank"] = ranked["Relationship_Strength"].apply(relationship_rank)
    ranked["ONET_Tier_Rank"] = ranked["ONET_Tier"].apply(onet_rank)
    ranked["Wage_Rank"] = ranked["Wage_Group"].apply(wage_sort_rank)

    ranked = (
        ranked
        .sort_values(
            [
                "ONET_Tier_Rank",
                "Relationship_Rank",
                "Score",
                "Wage_Rank",
                "Destination_Wage",
                "Destination_Growth",
                "Destination_Openings",
            ],
            ascending=[True, True, False, True, False, False, False]
        )
        .drop_duplicates(subset=["Destination_SOC", "Pathway_View", "Bridge_SOC"])
        .head(n)
        .copy()
    )

    return ranked



def make_sankey(edges, source_soc, source_title, mode):
    """
    Publishable Sankey layout.

    Design logic:
    1. Source occupation
    2. Pathway type or intermediary occupation
       - Direct pathways go through one clear "Direct related occupations" bucket.
       - Intermediary pathways show the actual stepping-stone occupation name.
    3. Wage comparison
    4. Lower-exposure destination occupations

    Color logic:
    - Source node: dark turquoise.
    - Direct pathway links: coral.
    - Intermediary pathway links: carrot.
    - Final destination links and destination nodes: exposure colors.
    """
    source_exposure = econ_value(source_soc, "Exposure")

    if mode == "direct":
        edges = edges[edges["Pathway_View"] == "direct"].copy()
        title_prefix = "Direct transition pathways"
        subtitle = "One-step lower-exposure options ranked by compatibility first, then wage."
    elif mode == "bridge":
        edges = edges[edges["Pathway_View"] == "bridge"].copy()
        title_prefix = "Intermediary transition pathways"
        subtitle = "Stepping-stone roles that lead to additional lower-exposure options."
    else:
        edges = edges.copy()
        title_prefix = "Direct and intermediary transition pathways"
        subtitle = "Ranked by compatibility first, then wage."

    if edges.empty:
        return None

    edges = select_top_destinations(edges, TOP_DESTINATIONS_PER_VIEW)

    if edges.empty:
        return None

    # Restore the pathway/intermediary layer so the chart shows the "wave" and the actual
    # direct/intermediary structure.
    edges = edges.copy()
    edges["Middle_Key"] = np.where(
        edges["Pathway_View"] == "direct",
        "direct_bucket",
        "bridge_" + edges["Bridge_SOC"].astype(str)
    )
    edges["Middle_Label"] = np.where(
        edges["Pathway_View"] == "direct",
        "Direct related occupations",
        edges["Bridge_Occupation"].fillna("").astype(str)
    )

    source_wage = econ_value(source_soc, "Median_Wage")
    wage_order = ["Higher wage", "Similar wage", "Lower wage", "Wage unknown"]

    middle_nodes = (
        edges[["Middle_Key", "Middle_Label", "Pathway_View", "Bridge_SOC", "Bridge_Occupation"]]
        .drop_duplicates()
        .copy()
    )
    middle_nodes["Sort_Order"] = middle_nodes["Pathway_View"].map({"direct": 1, "bridge": 2})
    middle_nodes = middle_nodes.sort_values(["Sort_Order", "Middle_Label"])

    wages = edges[["Wage_Group"]].drop_duplicates().copy()
    wages["Order"] = wages["Wage_Group"].map({v: i for i, v in enumerate(wage_order)})
    wages = wages.sort_values("Order")

    destinations = (
        edges[[
            "Destination_SOC",
            "Destination_Occupation",
            "Destination_AI_Exposure_Group",
            "Wage_Group",
        ]]
        .drop_duplicates(subset=["Destination_SOC"])
        .copy()
    )

    dest_sort = (
        edges[[
            "Destination_SOC",
            "Relationship_Strength",
            "ONET_Tier",
            "Score",
            "Destination_Wage",
            "Wage_Rank",
            "Destination_Growth",
            "Destination_Openings",
        ]]
        .drop_duplicates(subset=["Destination_SOC"])
        .copy()
    )

    dest_sort["ONET_Tier_Rank"] = dest_sort["ONET_Tier"].apply(onet_rank)
    dest_sort["Relationship_Rank"] = dest_sort["Relationship_Strength"].apply(relationship_rank)

    destinations = destinations.merge(
        dest_sort[[
            "Destination_SOC",
            "ONET_Tier_Rank",
            "Relationship_Rank",
            "Score",
            "Wage_Rank",
            "Destination_Wage",
            "Destination_Growth",
            "Destination_Openings",
        ]],
        on="Destination_SOC",
        how="left"
    )

    destinations = destinations.sort_values(
        [
            "ONET_Tier_Rank",
            "Relationship_Rank",
            "Score",
            "Wage_Rank",
            "Destination_Wage",
            "Destination_Growth",
            "Destination_Openings",
        ],
        ascending=[True, True, False, True, False, False, False]
    )

    node_labels, node_colors, node_x, node_y, node_hover = [], [], [], [], []
    node_lookup = {}

    def add_node(key, label, label_type, color, x, y, hover):
        if key not in node_lookup:
            node_lookup[key] = len(node_labels)
            node_labels.append(style_label(label, label_type))
            node_colors.append(color)
            node_x.append(x)
            node_y.append(y)
            node_hover.append(hover)
        return node_lookup[key]

    origin_id = add_node(
        "source",
        short(source_title, 30),
        "source",
        SOURCE_COLOR,
        0.02,
        0.50,
        build_hover_text(
            source_title,
            source_soc,
            exposure=source_exposure,
            role_label="Starting occupation"
        )
    )

    # Pathway/intermediary column
    middle_ys = make_positions(len(middle_nodes), top=0.34, bottom=0.70)

    for i, (_, row) in enumerate(middle_nodes.iterrows()):
        if row["Pathway_View"] == "direct":
            label = "Direct related<br>occupations"
            color = DIRECT_COLOR
            hover = (
                "<b>Direct related occupations</b><br>"
                "One-step transitions from the selected source occupation."
            )
        else:
            label = wrap_label(row["Middle_Label"], width=30, max_lines=3)
            color = INTERMEDIARY_COLOR
            hover = (
                f"<b>Intermediary occupation</b><br>"
                f"{row['Middle_Label']}<br>"
                "This stepping-stone occupation connects the source role to additional lower-exposure destinations."
            )

        add_node(
            f"middle_{row.Middle_Key}",
            label,
            "source",
            color,
            0.34,
            float(middle_ys[i]),
            hover
        )

    # Wage column
    wage_ys = make_positions(len(wages), top=0.28, bottom=0.76)

    for i, (_, row) in enumerate(wages.iterrows()):
        add_node(
            f"wage_{row.Wage_Group}",
            row.Wage_Group,
            "wage",
            WAGE_COLOR,
            0.65,
            float(wage_ys[i]),
            f"<b>{row.Wage_Group}</b><br>Source wage: {fmt_currency(source_wage)}"
        )

    # Destination column
    dest_ys = make_positions(len(destinations), top=0.13, bottom=0.80)
    dest_y_lookup = dict(zip(destinations["Destination_SOC"].astype(str), dest_ys))

    for _, row in destinations.iterrows():
        exposure = row["Destination_AI_Exposure_Group"]

        dest_subset = edges[
            edges["Destination_SOC"].astype(str) == str(row.Destination_SOC)
        ].copy()

        rel = dest_subset["Relationship_Strength"].iloc[0] if not dest_subset.empty else None
        src_type = dest_subset["Source_Type"].iloc[0] if not dest_subset.empty else None
        tier = dest_subset["ONET_Tier"].iloc[0] if not dest_subset.empty else None
        score = dest_subset["Score"].iloc[0] if not dest_subset.empty else None
        wage_group = dest_subset["Wage_Group"].iloc[0] if not dest_subset.empty else None
        pathway = dest_subset["Pathway_Label"].iloc[0] if not dest_subset.empty else None
        bridge = (
            dest_subset["Bridge_Occupation"].iloc[0]
            if not dest_subset.empty and dest_subset["Pathway_View"].iloc[0] == "bridge"
            else None
        )

        needs_more_edu = False

        dest_subset_check = edges[
            edges["Destination_SOC"].astype(str) == str(row.Destination_SOC)
        ]

        if not dest_subset_check.empty:
            needs_more_edu = bool(
                dest_subset_check["Requires_More_Education"].iloc[0]
            )

        education_dot = ""

        if needs_more_edu:
            education_dot = " ●"

        destination_label = (
            wrap_label(row.Destination_Occupation, width=28, max_lines=3)
            + education_dot
        )

        add_node(
            f"dest_{row.Destination_SOC}",
            destination_label,
            "destination",
            EXPOSURE_COLORS.get(exposure, "#DDDDDD"),
            0.96,
            float(dest_y_lookup[str(row.Destination_SOC)]),
            build_hover_text(
                row.Destination_Occupation,
                row.Destination_SOC,
                source_soc=source_soc,
                exposure=exposure,
                score=score,
                wage_group=wage_group,
                pathway=pathway,
                bridge=bridge,
                source_type=src_type,
                onet_tier=tier,
                relationship=rel,
                role_label="Destination occupation"
            )
        )

    sources, targets, values, colors, hovers = [], [], [], [], []

    # Source to pathway/intermediary links, colored by pathway type.
    grouped_middle = (
        edges.groupby(["Middle_Key", "Middle_Label", "Pathway_View"], as_index=False)
        .agg(
            Value=("Relationship_Weight", "sum"),
            Destinations=("Destination_SOC", "nunique")
        )
    )

    for _, row in grouped_middle.iterrows():
        path_color = DIRECT_COLOR if row["Pathway_View"] == "direct" else INTERMEDIARY_COLOR
        path_label = "Direct" if row["Pathway_View"] == "direct" else "Intermediary"

        sources.append(origin_id)
        targets.append(node_lookup[f"middle_{row.Middle_Key}"])
        values.append(row["Value"])
        colors.append(hex_to_rgba(path_color, 0.38))
        hovers.append(
            f"<b>{path_label} pathway</b><br>"
            f"{row.Middle_Label}<br>"
            f"Connected destinations: {row['Destinations']}"
        )

    # Pathway/intermediary to wage links, still colored by pathway type.
    grouped_middle_wage = (
        edges.groupby(["Middle_Key", "Middle_Label", "Pathway_View", "Wage_Group"], as_index=False)
        .agg(
            Value=("Relationship_Weight", "sum"),
            Destinations=("Destination_SOC", "nunique")
        )
    )

    for _, row in grouped_middle_wage.iterrows():
        path_color = DIRECT_COLOR if row["Pathway_View"] == "direct" else INTERMEDIARY_COLOR
        path_label = "Direct" if row["Pathway_View"] == "direct" else "Intermediary"

        sources.append(node_lookup[f"middle_{row.Middle_Key}"])
        targets.append(node_lookup[f"wage_{row.Wage_Group}"])
        values.append(row["Value"])
        colors.append(hex_to_rgba(path_color, 0.26))
        hovers.append(
            f"<b>{path_label} pathway</b><br>"
            f"{row.Middle_Label} to {row.Wage_Group}<br>"
            f"Destinations: {row['Destinations']}"
        )

    # Wage to destination links, colored by destination exposure.
    for _, row in edges.iterrows():
        exposure_color = EXPOSURE_COLORS.get(row["Destination_AI_Exposure_Group"], "#999999")

        sources.append(node_lookup[f"wage_{row.Wage_Group}"])
        targets.append(node_lookup[f"dest_{row.Destination_SOC}"])
        values.append(row["Relationship_Weight"])
        colors.append(hex_to_rgba(exposure_color, 0.48))
        hovers.append(
            build_hover_text(
                row.Destination_Occupation,
                row.Destination_SOC,
                source_soc=source_soc,
                exposure=row.Destination_AI_Exposure_Group,
                score=row.Score,
                wage_group=row.Wage_Group,
                pathway=row.Pathway_Label,
                bridge=row.Bridge_Occupation if row.Pathway_View == "bridge" else None,
                source_type=row.Source_Type,
                onet_tier=row.ONET_Tier,
                relationship=row.Relationship_Strength,
                role_label="Destination occupation"
            )
        )

    fig = go.Figure(data=[go.Sankey(
        arrangement="fixed",
        node=dict(
            pad=25,
            thickness=14,
            line=dict(color="white", width=1.4),
            label=node_labels,
            color=node_colors,
            x=node_x,
            y=node_y,
            customdata=node_hover,
            hovertemplate="%{customdata}<extra></extra>",
            hoverlabel=dict(
                bgcolor="white",
                bordercolor="#333333",
                font=dict(color="#111111", size=12, family="Arial"),
                align="left"
            )
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=colors,
            customdata=hovers,
            hovertemplate="%{customdata}<extra></extra>",
            hoverlabel=dict(
                bgcolor="white",
                bordercolor="#333333",
                font=dict(color="#111111", size=12, family="Arial"),
                align="left"
            )
        )
    )])

    annotations = [
        dict(
            x=0.02, y=1.15, xref="paper", yref="paper",
            showarrow=False, align="left",
            text=f"<b>{wrap_label(title_prefix + ' for ' + source_title, width=70, max_lines=2)}</b>",
            font=dict(size=20, color=DARK)
        ),
        dict(
            x=0.02, y=1.105, xref="paper", yref="paper",
            showarrow=False, align="left",
            text=subtitle,
            font=dict(size=14, color=DARK)
        ),
        dict(
            x=0.02, y=1.055, xref="paper", yref="paper",
            showarrow=False, align="left",
            text=(
                "<b>Legend:</b> "
                "<span style='color:#F06E6C'>■</span> Direct pathway &nbsp; "
                "<span style='color:#F79732'>■</span> Intermediary pathway &nbsp; "
                "<span style='color:#17C3B2'>■</span> Very Low destination &nbsp; "
                "<span style='color:#8ED3F5'>■</span> Low destination &nbsp; "
                "<span style='color:#75618C'>■</span> Medium destination &nbsp; "
                "<span style='color:#BB638F'>●</span> Higher education required"
            ),
            font=dict(size=12, color=DARK)
        ),
        dict(
            x=0.05, y=0.985, xref="paper", yref="paper",
            showarrow=False,
            text="<b>High or Very High occupation</b>",
            font=dict(size=15, color=DARK)
        ),
        dict(
            x=0.31, y=0.985, xref="paper", yref="paper",
            showarrow=False,
            text="<b>Pathway Type</b>",
            font=dict(size=15, color=DARK)
        ),
        dict(
            x=0.57, y=0.985, xref="paper", yref="paper",
            showarrow=False,
            text="<b>Wage comparison</b>",
            font=dict(size=15, color=DARK)
        ),
        dict(
            x=0.86, y=0.985, xref="paper", yref="paper",
            showarrow=False,
            text="<b>Lower-exposure occupations</b>",
            font=dict(size=15, color=DARK)
        ),
        dict(
            x=0.02, y=-0.115, xref="paper", yref="paper",
            showarrow=False, align="left",
            text=(
                "<b>Direct pathway:</b> a one-step transition from the selected source occupation to a related lower-exposure occupation."
                 "<br><br>"
                "<b>Intermediary pathway:</b> a transition that first moves through a related stepping-stone occupation, then connects to additional lower-exposure occupations."
                "<br><br>"
                "<b>Similar wage:</b> within ±5% for source wages under $50k, ±10% for $50k to $100k, and ±15% for source wages over $100k."
        ),
            font=dict(size=11.5, color="#333333")
        )
    ]

    fig.update_layout(
        annotations=annotations,
        autosize=True,
        width=None,
        height=690,
        font=dict(size=11, color="#111111", family="Arial"),
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(l=45, r=90, t=60, b=135),
        hoverlabel=dict(
            bgcolor="white",
            bordercolor="#333333",
            font=dict(color="#111111", size=12, family="Arial"),
            align="left"
        )
    )

    return fig

# ============================================================
# BUILD FIGURES FOR ALL SOURCES
# ============================================================

figure_store = {}

for _, src in source_options.iterrows():
    source_soc = str(src["Source_SOC"])
    source_title = clean_str(src["Source_Occupation"])

    edges = build_edges_for_source(source_soc)
    if edges.empty:
        continue

    # Find all Ferris titles/ids that share this same SOC
    matching_ferris_rows = ferris_occ[
        ferris_occ["soc"].astype(str) == source_soc
    ][["id", "title", "exposure"]].drop_duplicates()

    if matching_ferris_rows.empty:
        matching_ferris_rows = pd.DataFrame([{
            "id": make_id(source_title),
            "title": source_title,
            "exposure": clean_str(econ_value(source_soc, "Exposure"))
        }])

    for _, ferris_row in matching_ferris_rows.iterrows():
        source_id = ferris_row["id"]
        display_title = clean_str(ferris_row["title"])

        figure_store[source_id] = {
            "id": source_id,
            "soc": source_soc,
            "title": display_title,
            "exposure": clean_str(ferris_row["exposure"]),
            "both": None,
            "direct": None,
            "bridge": None,
        }

        for mode in ["both", "direct", "bridge"]:
            fig = make_sankey(edges, source_soc, display_title, mode)
            if fig is not None:
                figure_store[source_id][mode] = fig.to_plotly_json()

figure_store = {
    key: item
    for key, item in figure_store.items()
    if item["both"] is not None or item["direct"] is not None or item["bridge"] is not None
}

if not figure_store:
    raise ValueError("No Sankey figures were created.")

default_key = next(iter(figure_store))

# ============================================================
# WRITE LEAN HTML WITH SANKEY ONLY
# ============================================================

html = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>AI Transition Pathways Sankey</title>
    <script src="https://cdn.plot.ly/plotly-2.30.0.min.js"></script>

    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 12px 18px 32px 18px;
            color: #111111;
            background: white;
            overflow: hidden;
        }}
        
        .controls {{
            display: flex;
            justify-content: center;
            align-items: flex-end;
            gap: 28px;
            margin: 10px auto 6px auto;
            flex-wrap: nowrap;
            width: 100%;
        }}

        .control-group {{
            display: flex;
            flex-direction: column;
            align-items: center;
            text-align: center;
        }}

        label {{
            font-weight: bold;
            font-size: 14px;
        }}

        select {{
            padding: 8px 10px;
            font-size: 14px;
            border: 1px solid #cccccc;
            border-radius: 6px;
            background: white;
        }}

        #occupationSelect {{
            min-width: 450px;
        }}

        #viewSelect {{
            min-width: 260px;
        }}

        #chart {{
            width: 100%;
            min-height: 760px;
            overflow-y: hidden;
        }}
    </style>
</head>

<body>

<div class="controls">
    <div class="control-group">
        <label for="occupationSelect">Occupation Selected</label>
        <select id="occupationSelect" onchange="renderFigure()"></select>
    </div>

    <div class="control-group">
        <label for="viewSelect">Pathway view</label>
        <select id="viewSelect" onchange="renderFigure()">
            <option value="both">Both direct and intermediary pathways</option>
            <option value="direct">Direct related occupations only</option>
            <option value="bridge">Intermediary transition occupations only</option>
        </select>
    </div>
</div>

<div id="chart"></div>

<script>
const figures = {json.dumps(figure_store)};

const allOccupations = Object.entries(figures).map(([id, item]) => {{
    return {{
        id: id,
        soc: item.soc,
        title: item.title,
        exposure: item.exposure,
        label: item.title + " (" + item.soc + ", " + item.exposure + ")"
    }};
}});

function populateDropdown() {{
    const select = document.getElementById("occupationSelect");
    select.innerHTML = "";

    allOccupations.forEach(item => {{
        const opt = document.createElement("option");
        opt.value = item.id;
        opt.textContent = item.label;
        select.appendChild(opt);
    }});
}}

function renderFigure() {{
    const selectedId = document.getElementById("occupationSelect").value;
    let view = document.getElementById("viewSelect").value;
    const item = figures[selectedId];

    if (!item) return;

    if (!item[view]) {{
        if (item.both) view = "both";
        else if (item.direct) view = "direct";
        else view = "bridge";

        document.getElementById("viewSelect").value = view;
    }}

    const fig = JSON.parse(JSON.stringify(item[view]));

    fig.layout.autosize = true;
    fig.layout.width = null;
    fig.layout.height = 720;
    fig.layout.margin = {{ l: 55, r: 130, t: 190, b: 120 }};

fig.layout.annotations.forEach(a => {{
    if (a.text && a.text.includes("Ranked by")) {{
        a.text = "";
    }}

    if (a.text && a.text.includes("Direct and intermediary")) {{
        a.x = 0;
        a.y = 1.40;
        a.font.size = 22;
    }}

    if (a.text && a.text.includes("Legend")) {{
        a.y = 1.30;
        a.font.size = 12;
    }}

    if (a.text && a.text.includes("High or Very High")) {{
        a.x = 0;
        a.y = 1.10;
    }}

    if (a.text && a.text.includes("Pathway Type")) {{
        a.y = 1.10;
    }}

    if (a.text && a.text.includes("Wage comparison")) {{
        a.x = 0.69;
        a.y = 1.10;
    }}

    if (a.text && a.text.includes("Lower-exposure")) {{
        a.x = 1.00;
        a.y = 1.10;
    }}

        if (a.text && a.text.includes("Direct pathway:")) {{
        a.y = -0.20;
    }}
}});

    Plotly.react(
        "chart",
        fig.data,
        fig.layout,
        {{
            responsive: true,
            displayModeBar: true
        }}
    );
}}

populateDropdown();

const params = new URLSearchParams(window.location.search);
const requestedId = params.get("id");
const requestedSoc = params.get("soc");

let selectedId = "{default_key}";

if (requestedId && figures[requestedId]) {{
    selectedId = requestedId;
}} else if (requestedSoc) {{
    const matching = allOccupations.find(item => item.soc === requestedSoc);
    if (matching) {{
        selectedId = matching.id;
    }}
}}

document.getElementById("occupationSelect").value = selectedId;
renderFigure();


</script>

</body>
</html>
"""

OUT_HTML.parent.mkdir(parents=True, exist_ok=True)
OUT_HTML.write_text(html, encoding="utf-8")

print("Saved Sankey to:", OUT_HTML)
print("High and Very High AI occupations with figures:", len(figure_store))


Workbook sheets: ['Related_Occs_Final', 'Final_Transitions', 'Clean_Direct_Lower_Only', 'Clean_Gateway_Pathways', 'Lightcast_Parsed', 'ONET_Parsed', 'Lightcast_Raw_All', 'Lightcast_Source_Econ', 'Lightcast_Source_Edu', 'AI_Economic', 'Score_Summary', 'Source_Type_By_Source', 'Target_Not_In_AI', 'Lightcast_Parse_Errors', 'SOC_Title_Lookup', 'Crosswalk_Used', 'Recommended_Direct_Only', 'Recommended_Gateway_Top3', 'Direct_Source_Summary_v2', 'Gateway_Source_Summary_v2']
Pathway sheet used: Related_Occs_Final
Rows in pathway sheet: 1128
Rows in AI_Economic: 825
OEWS wage fallback applied.
Missing wages before OEWS fallback: 126
Missing wages after OEWS fallback: 126
High and Very High source occupations included: 210
Saved Sankey to: C:\Users\301533\Documents\GitHub\AI\AI Website\aipathways.github.io\assets\sankey.html
High and Very High AI occupations with figures: 296
